<a href="https://colab.research.google.com/github/techexplorer0123-tech/vla-simulation-research/blob/main/01_first_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SmolVLA on LIBERO: baseline eval and custom-prompt rollouts

1. Install LeRobot with the LIBERO simulator on a Colab GPU runtime.
2. Evaluate the pretrained `lerobot/smolvla_libero` checkpoint on one LIBERO-Spatial task (baseline).
3. Re-run the same task with a custom language instruction and record the episode to a local LeRobot dataset.

## 1. Environment

In [ ]:
!nvidia-smi -L
import sys, torch
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# System libraries: EGL/OSMesa for headless MuJoCo rendering, ffmpeg for video encoding,
# cmake/build-essential to compile egl_probe from source.
!apt-get update -qq && apt-get install -y -qq git ffmpeg libgl1 libglib2.0-0 libosmesa6 libegl1 \
    cmake build-essential python3-dev pkg-config > /dev/null

In [ ]:
# Set once, before any MuJoCo import. Every `!`, `%pip` and subprocess below inherits os.environ.
import os
os.environ["MUJOCO_GL"] = "egl"                     # headless rendering for MuJoCo / LIBERO
os.environ["CMAKE_POLICY_VERSION_MINIMUM"] = "3.5"  # egl_probe's CMakeLists predates CMake 4

In [ ]:
# hf-libero depends on hf-egl-probe and on robomimic==0.2.0 (-> egl_probe); both are source-only
# tarballs whose CMake build fails inside pip's isolated build env. Build them against the system
# toolchain first, otherwise `pip install lerobot[libero]` fails and no CLI entry points get installed.
%pip install -q "cmake<4"
%pip install -q --no-build-isolation --no-cache-dir egl_probe==1.0.2 hf-egl-probe==1.0.2

In [ ]:
# `libero` already includes the dataset + video (av) deps, so the `evaluation` extra is not needed.
!git clone -q --depth 1 https://github.com/huggingface/lerobot.git /content/lerobot
%pip install -q -e "/content/lerobot[smolvla,libero]"

In [ ]:
# Smoke test: headless MuJoCo rendering through EGL.
import mujoco
from PIL import Image

xml = """<mujoco><worldbody>
  <light pos="0 0 2"/><geom type="plane" size="1 1 0.1"/>
  <body pos="0 0 0.2"><geom type="sphere" size="0.1" rgba="0.2 0.6 0.9 1"/></body>
</worldbody></mujoco>"""
model = mujoco.MjModel.from_xml_string(xml)
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)
with mujoco.Renderer(model, height=240, width=320) as renderer:
    renderer.update_scene(data)
    display(Image.fromarray(renderer.render()))

## 2. Shared experiment settings

In [ ]:
import json, shlex
from datetime import datetime
from pathlib import Path

POLICY = "lerobot/smolvla_libero"
POLICY_REVISION = "a24f1f380461b665158d52acd140152c79e8b36d"
# The checkpoint was trained with camera1/camera2; the LIBERO env emits image/image2.
RENAME_MAP = json.dumps({"observation.images.image": "observation.images.camera1",
                         "observation.images.image2": "observation.images.camera2"})
TASK_SUITE, TASK_ID = "libero_spatial", 0
FPS = 20  # LIBERO's control rate; --fps and --dataset.fps must both equal it
RESULTS = Path("/content/vla_results")


def run(cmd: list[str]) -> None:
    """Run a CLI command with live output in the cell (subprocess.run would hide it)."""
    get_ipython().system(shlex.join(cmd))

## 3. Baseline: `lerobot-eval` on the task's own instruction

In [ ]:
run([
    "lerobot-eval",
    f"--policy.path={POLICY}",
    f"--policy.pretrained_revision={POLICY_REVISION}",
    "--policy.device=cuda",
    "--env.type=libero",
    f"--env.task={TASK_SUITE}",
    f"--env.task_ids=[{TASK_ID}]",
    f"--rename_map={RENAME_MAP}",
    "--eval.n_episodes=1",
    "--eval.batch_size=1",
    f"--output_dir={RESULTS / 'baseline'}",
])

## 4. Custom prompt: `lerobot-rollout` with recording

The episodic strategy restarts the simulated episode, runs the policy on `CUSTOM_PROMPT`, stops when the
environment ends the episode (success, failure or step limit) or after `episode_time_s`, and writes the
frames to a local LeRobot dataset with videos.

In [ ]:
PROMPT_ID = "P004"
CUSTOM_PROMPT = "Pick up the black bowl and place the ramekin on the plate"
# Fresh directory per run so earlier recordings for the same prompt are kept.
RUN_DIR = RESULTS / "custom_prompts" / PROMPT_ID / datetime.now().strftime("%Y%m%d_%H%M%S")

run([
    "lerobot-rollout",
    "--strategy.type=episodic",
    "--strategy.reset_to_initial_position=false",
    "--robot.type=sim",
    "--robot.env.type=libero",
    f"--robot.env.task={TASK_SUITE}",
    f"--robot.task_id={TASK_ID}",
    f"--task={CUSTOM_PROMPT}",
    f"--policy.path={POLICY}",
    f"--policy.pretrained_revision={POLICY_REVISION}",
    f"--rename_map={RENAME_MAP}",
    "--device=cuda",
    f"--fps={FPS}",
    "--inference.type=sync",
    "--display_data=false",
    "--return_to_initial_position=false",
    "--play_sounds=false",
    f"--dataset.repo_id=local/rollout_{PROMPT_ID.lower()}",
    f"--dataset.root={RUN_DIR}",
    f"--dataset.fps={FPS}",
    "--dataset.num_episodes=1",
    # Wall-clock cap. The sim loop runs at ~3.5 Hz on Colab (sim step + rendering, not GPU),
    # so 60 s would stop the episode at ~210 steps, before LIBERO's own 280-step limit.
    # 150 s lets the environment, not the timer, end the episode.
    "--dataset.episode_time_s=150",
    "--dataset.reset_time_s=0",
    "--dataset.push_to_hub=false",
])

In [ ]:
from IPython.display import Video

videos = sorted(RUN_DIR.rglob("*.mp4"))
print("Prompt:", CUSTOM_PROMPT)
print(*videos, sep="\n")
Video(videos[0], embed=True, width=480) if videos else print("No video recorded; check the rollout log above.")